# 03 — Causal results (ADR-008)

This notebook **only reads** `data/processed/results/` (written by `analysis/causal.py`, run with `make analysis`). It estimates nothing. Check the `status` column: results are **PROVISIONAL** until every control station has predictions.

In [1]:
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'analysis' else Path.cwd()
RES = ROOT / 'data' / 'processed' / 'results'
FIG = ROOT / 'docs' / 'figures'
pd.set_option('display.max_rows', 200, 'display.width', 200)
pd.read_parquet(RES / 'run_meta.parquet')

,status,control_stations_with_predictions,control_stations_in_study,panel_station_days,runtime_s
0,PROVISIONAL,191,587,976897,98


## Every estimate

In [2]:
est = pd.read_parquet(RES / 'estimates.parquet')
est[['status', 'family', 'spec', 'outcome', 'estimate', 'ci_low', 'ci_high', 'n_stations', 'n_station_days']].round(3)

,status,family,spec,outcome,estimate,ci_low,ci_high,n_stations,n_station_days
0,PROVISIONAL,primary,€9-Ticket: triple difference (Jun–Aug vs Jan–M...,ratio_pct,5.502,2.877,8.127,457,329333
1,PROVISIONAL,primary,€9-Ticket: triple difference (Jun–Aug vs Jan–M...,resid_ugm3,0.002,-0.760,0.763,457,329335
2,PROVISIONAL,secondary (a),switch-off: Sep–Dec vs Jan–May 2022,ratio_pct,6.729,4.007,9.451,457,370662
3,PROVISIONAL,secondary (a),switch-off: Sep–Dec vs Jan–May 2022,resid_ugm3,1.114,0.257,1.971,457,370663
4,PROVISIONAL,secondary (a),"switch-off: Sep–Dec vs Jan–May 2022, controls ...",ratio_pct,9.608,4.597,14.619,340,276790
5,PROVISIONAL,secondary (a),"switch-off: Sep–Dec vs Jan–May 2022, controls ...",resid_ugm3,2.205,0.866,3.544,340,276790
6,PROVISIONAL,secondary (b),Deutschlandticket: May–Dec vs Jan–Apr 2023,ratio_pct,-0.082,-2.278,2.114,457,495213
7,PROVISIONAL,secondary (b),Deutschlandticket: May–Dec vs Jan–Apr 2023,resid_ugm3,-0.723,-1.399,-0.047,457,495217
8,PROVISIONAL,secondary (b),persistence: 2024 vs Jan–Apr 2023,ratio_pct,-1.397,-3.025,0.230,457,213742
9,PROVISIONAL,secondary (b),persistence: 2024 vs Jan–Apr 2023,resid_ugm3,-0.372,-0.923,0.180,457,213744


## Wild cluster bootstrap (primary)

In [3]:
pd.read_parquet(RES / 'wild_bootstrap.parquet')

,spec,estimate,se_country_crv,p_value,ci_low,ci_high,ci_at_grid_edge,n_clusters,n_boot,weights,seed,status
0,primary,5.502032,1.759628,0.044404,0.135168,11.924673,False,8,9999,Webb,42,PROVISIONAL


## Placebos

![placebos](../docs/figures/placebos.png)

## Event study

![event study](../docs/figures/event_study.png)

## Synthetic control

![synthetic control](../docs/figures/synthetic_control.png)

In [4]:
pd.read_parquet(RES / 'sc_weights.parquet').pivot(index='donor', columns='variant', values='weight').round(3)

variant,demeaned,raw
donor,,
AT,0.154,0.154
BE,0.431,0.434
CH,0.118,0.120
CZ,0.000,0.000
FR,0.091,0.089
NL,0.206,0.203
PL,0.000,0.000


In [5]:
pd.read_parquet(RES / 'sc_rmspe.parquet').round(3)

,variant,unit,rmspe_pre,rmspe_post Jun–Aug 2022,ratio Jun–Aug 2022,mean_gap Jun–Aug 2022,rmspe_post May–Dec 2023,ratio May–Dec 2023,mean_gap May–Dec 2023,rank Jun–Aug 2022,p Jun–Aug 2022,rank May–Dec 2023,p May–Dec 2023,status
0,raw,DE,2.776,1.510,0.544,0.123,3.168,1.141,-2.655,8.0,1.000,3.0,0.375,PROVISIONAL
1,raw,AT,4.136,6.656,1.609,-5.771,4.885,1.181,0.048,2.0,0.250,2.0,0.250,PROVISIONAL
2,raw,BE,3.731,6.975,1.870,-6.902,3.989,1.069,-3.043,1.0,0.125,4.0,0.500,PROVISIONAL
3,raw,CH,4.559,5.600,1.228,4.438,4.800,1.053,0.878,4.0,0.500,6.0,0.750,PROVISIONAL
4,raw,CZ,4.676,5.749,1.229,3.000,4.916,1.051,3.896,3.0,0.375,7.0,0.875,PROVISIONAL
5,raw,FR,2.857,2.502,0.876,2.240,3.786,1.325,-0.234,6.0,0.750,1.0,0.125,PROVISIONAL
6,raw,NL,2.553,1.626,0.637,1.173,2.724,1.067,-1.420,7.0,0.875,5.0,0.625,PROVISIONAL
7,raw,PL,4.915,4.650,0.946,2.631,2.720,0.553,0.337,5.0,0.625,8.0,1.000,PROVISIONAL
8,demeaned,DE,2.775,1.501,0.541,0.027,3.237,1.167,-2.738,8.0,1.000,3.0,0.375,PROVISIONAL
9,demeaned,AT,4.136,6.680,1.615,-5.799,4.884,1.181,0.020,2.0,0.250,2.0,0.250,PROVISIONAL


## Country map data

In [6]:
pd.read_parquet(RES / 'country_map.parquet').round(2)

,window,country_code,value,n_stations,lat,lon,status
0,Jun–Aug 2022 vs Jan–May 2022,AT,-1.28,53,47.79,15.45,PROVISIONAL
1,Jun–Aug 2022 vs Jan–May 2022,BE,6.13,19,51.00,4.42,PROVISIONAL
2,Jun–Aug 2022 vs Jan–May 2022,CH,4.18,3,47.54,7.96,PROVISIONAL
3,Jun–Aug 2022 vs Jan–May 2022,CZ,-0.09,7,50.55,14.80,PROVISIONAL
4,Jun–Aug 2022 vs Jan–May 2022,DE,8.50,284,50.91,9.98,PROVISIONAL
5,Jun–Aug 2022 vs Jan–May 2022,FR,6.80,44,48.09,3.65,PROVISIONAL
6,Jun–Aug 2022 vs Jan–May 2022,NL,5.36,34,52.10,4.95,PROVISIONAL
7,Jun–Aug 2022 vs Jan–May 2022,PL,-1.31,13,50.39,18.43,PROVISIONAL
8,May–Dec 2023 vs Jan–Apr 2023,AT,2.29,53,47.79,15.45,PROVISIONAL
9,May–Dec 2023 vs Jan–Apr 2023,BE,3.94,19,51.00,4.42,PROVISIONAL
